In [7]:
# System Imports
import sys
sys.path.insert(0, "../src")
import codebookmap
import armpipelines

# project imports
from codebookmap import BRFSS_MAP
from armpipelines import make_arm_a
from armpipelines import make_arm_b

#pandas and numpy import
import pandas as pd
import numpy as np

# silence warnings
import warnings
warnings.filterwarnings('ignore')

# Pipeline and simple imputer imports
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [13]:
dev = pd.read_parquet("../../data/dev.parquet")
dev.head()

,_STATE,_METSTAT,_URBSTAT,SEXVAR,_AGE80,_RACE,MARITAL,EDUCA,RENTHOM1,VETERAN3,...,FRENCHF1,POTATOE1,VEGETAB2,FRUTDA2_,FTJUDA2_,GRENDA1_,FRNCHDA_,POTADA1_,VEGEDA2_,CLASS
0,33.0,2.0,2.0,2.0,54.0,1.0,1.0,6.0,1.0,1.0,...,555.0,301.0,202.0,43.0,4.300000e+01,43.0,5.397605e-79,3.000000e+00,29.0,1
1,39.0,1.0,1.0,1.0,59.0,1.0,1.0,4.0,1.0,1.0,...,203.0,306.0,203.0,43.0,5.700000e+01,29.0,4.300000e+01,2.000000e+01,43.0,1
2,19.0,1.0,1.0,1.0,74.0,1.0,1.0,4.0,1.0,1.0,...,300.0,300.0,101.0,33.0,5.397605e-79,100.0,2.000000e+00,2.000000e+00,100.0,1
3,36.0,1.0,1.0,2.0,46.0,2.0,1.0,6.0,1.0,2.0,...,201.0,101.0,101.0,200.0,5.397605e-79,100.0,1.400000e+01,1.000000e+02,100.0,0
4,47.0,2.0,1.0,1.0,63.0,1.0,2.0,5.0,1.0,1.0,...,102.0,555.0,101.0,14.0,1.400000e+01,200.0,2.000000e+02,5.397605e-79,100.0,0


In [20]:
summary_stats = dev['FRUTDA2_'].describe()
print(summary_stats)

desc_with_na = summary_stats.append(pd.Series({
    'NaN_count': dev['PHYSHLTH'].isna().sum()
}))
print(desc_with_na)

count    4.421200e+04
mean     1.211596e+02
std      3.805122e+02
min      5.397605e-79
25%      4.300000e+01
50%      1.000000e+02
75%      1.000000e+02
max      9.900000e+03
Name: FRUTDA2_, dtype: float64
count        4.421200e+04
mean         1.211596e+02
std          3.805122e+02
min          5.397605e-79
25%          4.300000e+01
50%          1.000000e+02
75%          1.000000e+02
max          9.900000e+03
NaN_count    0.000000e+00
dtype: float64


In [17]:
df_arm_b_X, df_arm_b_y = make_arm_b(dev)
df_arm_a_X, df_arm_a_y = make_arm_a(dev)

ValueError: could not convert string to float: 'Refused'

In [ ]:
#Pipeline A
# Numeric branch (Arm A)
naive_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean', add_indicator=False)),
    ('scaler', StandardScaler())
])

# Nominal branch (Arm A)
naive_nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# preprocessor (Arm A)
preprocessor_arm_a = ColumnTransformer(
    transformers=[
        ('num', naive_numeric_transformer, numeric_cols),
        ('cat', naive_nominal_transformer, nominal_cols)
    ]
)

In [ ]:
#Pipeline B
#  Rule 1(Arm B) : One-Hot Encode (including 'Unknown')
nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

#  Rule 2 (Arm B): Impute -> Missing Indicators -> Standardize
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

# preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', nominal_transformer, nominal_cols)
    ]
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, auc
